[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/cancel_test.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib

# Cancel test

This example corresponds to the `cancel_test` sample, a port of CancelTest of plhamJ.
`CancelFCNAgent` submits orders in the same way as `FCNAgent` and cancels each of them with the probability `cancelRate` (0.3 by default) right after submitting it.
An order that is executed as soon as it is placed is not affected by its cancel order; the other canceled orders are removed from the order book, so fewer orders rest in the book.

In [ ]:
import copy
import random
from typing import Any
from typing import Dict
from typing import List
from typing import Union

import matplotlib.pyplot as plt
import numpy as np

from pams import Cancel
from pams import Market
from pams import Order
from pams.agents import FCNAgent
from pams.logs import CancelLog
from pams.logs import ExecutionLog
from pams.logs import MarketStepEndLog
from pams.logs import OrderLog
from pams.logs.market_step_loggers import MarketStepSaver
from pams.runners import SequentialRunner
from pams.utils import JsonRandom

In [ ]:
class CancelFCNAgent(FCNAgent):
    def setup(
        self,
        settings: Dict[str, Any],
        accessible_markets_ids: List[int],
        *args: Any,
        **kwargs: Any,
    ) -> None:
        super().setup(settings=settings, accessible_markets_ids=accessible_markets_ids)
        if "cancelRate" in settings:
            json_random: JsonRandom = JsonRandom(prng=self.prng)
            self.cancel_rate = json_random.random(json_value=settings["cancelRate"])
        else:
            self.cancel_rate = 0.3
        if not 0.0 <= self.cancel_rate <= 1.0:
            raise ValueError("cancelRate have to be between 0.0 and 1.0")

    def submit_orders_by_market(self, market: Market) -> List[Union[Order, Cancel]]:
        orders: List[Union[Order, Cancel]] = super().submit_orders_by_market(
            market=market
        )
        # the cancel orders follow all the orders, so that each order is placed before it is canceled
        cancels: List[Union[Order, Cancel]] = []
        for order in orders:
            if isinstance(order, Order) and self.prng.random() < self.cancel_rate:
                cancels.append(Cancel(order=order))
        return orders + cancels

In [ ]:
class CancelTestSaver(MarketStepSaver):
    def __init__(self) -> None:
        super().__init__()
        self.n_orders: int = 0
        self.n_cancels: int = 0
        self.n_effective_cancels: int = 0
        self.n_executions: int = 0

    def process_order_log(self, log: OrderLog) -> None:
        self.n_orders += 1

    def process_cancel_log(self, log: CancelLog) -> None:
        self.n_cancels += 1
        # the remaining volume is 0 if the order has been executed before it is canceled
        if log.volume > 0:
            self.n_effective_cancels += 1

    def process_execution_log(self, log: ExecutionLog) -> None:
        self.n_executions += 1

    def process_market_step_end_log(self, log: MarketStepEndLog) -> None:
        super().process_market_step_end_log(log)
        self.market_step_logs[-1]["n_book_orders"] = len(log.market.buy_order_book) + len(log.market.sell_order_book)

In [ ]:
config = {
	"simulation": {
		"markets": ["Market"],
		"agents": ["CancelFCNAgents"],
		"sessions": [
			{	"sessionName": 0,
				"iterationSteps": 100,
				"withOrderPlacement": True,
				"withOrderExecution": False,
				"withPrint": True
			},
			{	"sessionName": 1,
				"iterationSteps": 500,
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True
			}
		]
	},

	"Market": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"CancelFCNAgents": {
		"class": "CancelFCNAgent",
		"numAgents": 100,

		"MEMO": "Agent class",
		"markets": ["Market"],
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [1.0]},
		"chartWeight": {"expon": [0.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"orderMargin": [0.0, 0.1],

		"MEMO": "CancelFCNAgent class",
		"cancelRate": 0.3
	}
}

In [ ]:
def run(config: Dict, seed: int) -> CancelTestSaver:
    saver = CancelTestSaver()
    runner = SequentialRunner(
        settings=config,
        prng=random.Random(seed),
        logger=saver,
    )
    runner.class_register(cls=CancelFCNAgent)
    runner.main()
    return saver

In [ ]:
saver = run(config=config, seed=42)

In [ ]:
market_prices = dict(sorted(map(lambda x: (x["market_time"], x["market_price"]), saver.market_step_logs)))
fundamental_prices = dict(sorted(map(lambda x: (x["market_time"], x["fundamental_price"]), saver.market_step_logs)))

In [ ]:
plt.plot(list(market_prices.keys()), list(market_prices.values()), color='red')
plt.plot(list(fundamental_prices.keys()), list(fundamental_prices.values()), color='black')
plt.xlabel("ticks")
plt.ylabel("market price")
plt.ylim([270, 330])
plt.show()

About 30% of the orders are canceled. Only the effective cancels, i.e., those of the orders not executed yet, remove orders from the order book.

In [ ]:
print(f"orders: {saver.n_orders}")
print(f"cancels: {saver.n_cancels} ({saver.n_cancels / saver.n_orders:.1%} of the orders)")
print(f"effective cancels: {saver.n_effective_cancels} ({saver.n_effective_cancels / saver.n_orders:.1%} of the orders)")
print(f"executions: {saver.n_executions}")

## Comparison with no cancel

In [ ]:
config_no_cancel = copy.deepcopy(config)
config_no_cancel["CancelFCNAgents"]["cancelRate"] = 0.0
saver_no_cancel = run(config=config_no_cancel, seed=42)

In [ ]:
for label, _saver in [("cancelRate = 0.3", saver), ("cancelRate = 0.0", saver_no_cancel)]:
    n_book_orders = dict(sorted(map(lambda x: (x["market_time"], x["n_book_orders"]), _saver.market_step_logs)))
    plt.plot(list(n_book_orders.keys()), list(n_book_orders.values()), label=label)
plt.xlabel("ticks")
plt.ylabel("number of orders in the order book")
plt.legend()
plt.show()

## Cancel rate and market

Each cancel rate is simulated with 5 seeds, and the statistics of the second session (ticks 100-599) are averaged over the seeds.
The higher the cancel rate is, the fewer orders rest in the order book and the fewer orders are executed.
The thinner order book makes the market price more volatile.

In [ ]:
cancel_rates = [0.0, 0.2, 0.4, 0.6, 0.8]
seeds = range(5)
mean_book_orders = []
n_executions = []
volatilities = []
for cancel_rate in cancel_rates:
    _config = copy.deepcopy(config)
    _config["CancelFCNAgents"]["cancelRate"] = cancel_rate
    _mean_book_orders = []
    _n_executions = []
    _volatilities = []
    for seed in seeds:
        _saver = run(config=_config, seed=seed)
        logs = list(filter(lambda x: x["session_id"] == 1, _saver.market_step_logs))
        _mean_book_orders.append(np.mean(list(map(lambda x: x["n_book_orders"], logs))))
        _n_executions.append(_saver.n_executions)
        log_returns = np.diff(np.log(list(map(lambda x: x["market_price"], logs))))
        _volatilities.append(np.std(log_returns))
    mean_book_orders.append(np.mean(_mean_book_orders))
    n_executions.append(np.mean(_n_executions))
    volatilities.append(np.mean(_volatilities))

In [ ]:
for cancel_rate, mean_book_order, n_execution, volatility in zip(cancel_rates, mean_book_orders, n_executions, volatilities):
    print(f"cancelRate = {cancel_rate}: orders in the order book {mean_book_order:.1f}, executions {n_execution:.1f}, volatility of log returns {volatility:.5f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(cancel_rates, mean_book_orders, marker="o")
axes[0].set_xlabel("cancel rate")
axes[0].set_ylabel("mean number of orders in the order book")
axes[1].plot(cancel_rates, n_executions, marker="o")
axes[1].set_xlabel("cancel rate")
axes[1].set_ylabel("number of executions")
axes[2].plot(cancel_rates, volatilities, marker="o")
axes[2].set_xlabel("cancel rate")
axes[2].set_ylabel("volatility of log returns")
fig.tight_layout()
plt.show()